# Module 5 — Portfolios and a regression


The last step of the course combines everything: build a portfolio return series out of
individual stocks, turn it into a growth path, then explain one return series with
another using a regression.

`fin400-port-beta.csv` has monthly returns on ten portfolios sorted by past beta
(`p0` … `p9`); `fin400-factors.csv` has the market return `mkt` and a small-cap factor
`small`. Returns are decimals: `0.01` is one percent.


## 1. Equal weight first


In [1]:
import pandas as pd

stocks = pd.read_csv("fin400-monthly-stocks.csv", parse_dates=["caldt"])

ew = stocks.groupby("caldt")["ret"].mean()
print(len(ew))
print(round(float(ew.mean()), 4))
print(round(float(ew.std()), 4))


120
0.0086
0.0517


One number per year-month. `std()` in pandas divides by n − 1, the sample version the
class uses.


## 2. Value weight, and the growth path


In [2]:
stocks["weighted"] = stocks["ret"] * stocks["mcap"]
vw = stocks.groupby("caldt")["weighted"].sum() / stocks.groupby("caldt")["mcap"].sum()

growth = (1 + vw).cumprod() - 1
print(round(float(growth.iloc[-1]), 4))


4.9841


`cumprod()` compounds the monthly factors, so each entry in `growth` is the return from
the first month of the sample through that month. Subtracting 1 turns a growth factor
back into a return.


## 3. One regression


In [3]:
import statsmodels.formula.api as smf

port = pd.read_csv("fin400-port-beta.csv", parse_dates=["caldt"])
factors = pd.read_csv("fin400-factors.csv", parse_dates=["caldt"])

data = pd.merge(port, factors, on="caldt", how="inner")
capm = smf.ols("p9 ~ mkt", data=data).fit()

print(int(capm.nobs))
print(capm.params.round(4))


120
Intercept    0.0019
mkt          1.3451
dtype: float64


In [4]:
print(capm.summary())


                            OLS Regression Results                            
Dep. Variable:                     p9   R-squared:                       0.956
Model:                            OLS   Adj. R-squared:                  0.956
Method:                 Least Squares   F-statistic:                     2567.
Date:                Mon, 26 Oct 2026   Prob (F-statistic):           6.39e-82
Time:                        18:06:51   Log-Likelihood:                 363.81
No. Observations:                 120   AIC:                            -723.6
Df Residuals:                     118   BIC:                            -718.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0019      0.001      1.735      0.0

The formula string reads "left-hand side ~ right-hand side". `params` holds the fitted
intercept and slope; the summary adds standard errors, t-statistics and p-values.


## 4. Testing a value, not just zero


In [5]:
test = capm.t_test("mkt = 1")
print(test)
print(round(float(test.pvalue.squeeze()), 6))


                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             1.3451      0.027     12.999      0.000       1.293       1.398
0.0


The default p-value in the summary tests "slope = 0". `t_test` lets you ask about any
value — here, whether the beta of the highest-beta portfolio is really 1. A small
p-value rejects that claim.


## Try it

1. Regress each of `p0` … `p4` in `fin400-port-pe.csv` on `mkt` and `small`, keeping the
   results in a dictionary.
2. Test whether the intercept above equals −0.004.
3. Plot `growth` with `growth.plot()` and describe what the last value means.

**Practice 17**, **Practice 18** and **Practice 19** on the practice page check these.
